# ETAPA 1 - TRATAMENTO E PREPARAÇÃO DOS DADOS

## Projeto: análise e previsão do NPS

**Membros do Grupo:** 
- Amanda Alves de Lima Santos - RM376932 
- Fabio da Silva Costa - 
- Gabriel Victor Santos Oliveira
- Vanessa Partala

**Etapas do CRISP-DM:** compreensão e preparação dos dados  
**Arquivo de entrada:** `data/desafio_nps_fase_1.csv`  
**Arquivo de saída:** `data/processed/df_avaliacao.xlsx` e `nps_registros_rejeitados.csv"`

---
# BLOCO 01 - CONTEXTO E PREPARAÇÃO

## Contexto

A empresa possui informações sobre clientes, pedidos, entregas, atendimento e
satisfação. Antes da análise exploratória, precisamos verificar se
os dados estão completos, consistentes e adequados ao objetivo do projeto.

Este notebook foi organizado para que outras pessoas consigam acompanhar o
processo, entender as decisões tomadas e reproduzir os resultados.

## Objetivos

- Compreender a estrutura e a unidade de análise da base;
- Verificar valores ausentes, duplicidades e tipos de dados;
- Avaliar a coerência dos valores;
- Investigar possíveis outliers;
- Padronizar nomes e categorias;
- Criar variáveis derivadas úteis para a EDA;
- Classificar o NPS em Detrator, Neutro e Promotor;
- Salvar a base tratada em `data/processed/df_avaliacao.xlsx`.

## 1. Preparação do ambiente

Importamos apenas as bibliotecas necessárias para leitura, tratamento,
visualização e exportação dos dados.

In [1]:
# Manipulação e análise de dados
import pandas as pd

# Operações numéricas e criação de condições
import numpy as np

# Criação de gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Criação da pasta de saída
import os

## 2. Carregamento da base

A base original será carregada da pasta `data`.

O tratamento será realizado em uma cópia chamada `df`. Assim, o DataFrame
`df_original` permanece disponível para comparação ao final do processo.

In [2]:
# Caminho do arquivo de entrada, considerando o notebook dentro de notebooks/
caminho_entrada = "../data/desafio_nps_fase_1.csv"

# Leitura da base original
df_original = pd.read_csv(caminho_entrada)

# Cópia utilizada durante o tratamento
df = df_original.copy()

# Dimensão inicial da base
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

Quantidade de linhas: 2500
Quantidade de colunas: 19


### Resultado observado

A base foi carregada com **2.500 registros e 19 colunas**.

Cada linha reúne dados de um cliente e de um pedido. Como `order_id` identifica
a transação registrada, a unidade de análise adotada neste projeto é o **pedido**.

## 3. Reconhecimento da estrutura dos dados

Antes de alterar a base, observamos:

- As primeiras linhas;
- Os nomes das colunas;
- Os tipos armazenados;
- A quantidade de valores preenchidos;
- As estatísticas descritivas das variáveis numéricas.

In [3]:
# Visualização das primeiras linhas
df.head()

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


In [4]:
# Lista de colunas disponíveis
df.columns.tolist()

['customer_id',
 'customer_age',
 'customer_region',
 'customer_tenure_months',
 'order_id',
 'order_value',
 'items_quantity',
 'discount_value',
 'payment_installments',
 'delivery_time_days',
 'delivery_delay_days',
 'freight_value',
 'delivery_attempts',
 'customer_service_contacts',
 'resolution_time_days',
 'nps_score',
 'repeat_purchase_30d',
 'complaints_count',
 'csat_internal_score']

In [5]:
# Tipos das colunas e quantidade de valores não nulos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                2500 non-null   int64  
 1   customer_age               2500 non-null   int64  
 2   customer_region            2500 non-null   object 
 3   customer_tenure_months     2500 non-null   int64  
 4   order_id                   2500 non-null   int64  
 5   order_value                2500 non-null   float64
 6   items_quantity             2500 non-null   int64  
 7   discount_value             2500 non-null   float64
 8   payment_installments       2500 non-null   int64  
 9   delivery_time_days         2500 non-null   int64  
 10  delivery_delay_days        2500 non-null   int64  
 11  freight_value              2500 non-null   float64
 12  delivery_attempts          2500 non-null   int64  
 13  customer_service_contacts  2500 non-null   int64

In [6]:
# Estatísticas descritivas das variáveis numéricas
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
customer_id,2500.0,1250.50,721.83,1.00,625.75,1250.50,1875.25,2500.00
customer_age,2500.0,43.40,14.89,18.00,31.00,43.00,56.00,69.00
customer_tenure_months,2500.0,61.32,34.48,1.00,31.00,62.00,91.00,119.00
order_id,2500.0,51250.50,721.83,50001.00,50625.75,51250.50,51875.25,52500.00
order_value,2500.0,434.26,289.77,7.76,220.24,375.52,577.29,1983.81
items_quantity,2500.0,3.47,1.69,1.00,2.00,3.00,5.00,6.00
discount_value,2500.0,29.75,29.23,0.02,8.88,20.94,40.83,230.33
payment_installments,2500.0,6.00,3.16,1.00,3.00,6.00,9.00,11.00
delivery_time_days,2500.0,8.02,3.77,2.00,5.00,8.00,11.00,14.00
delivery_delay_days,2500.0,2.19,1.45,0.00,1.00,2.00,3.00,8.00


### Resultado observado

A base contém:

- 13 colunas do tipo inteiro;
- 5 colunas do tipo decimal;
- 1 coluna categórica, `customer_region`.

Os valores de `customer_age` variam de **18 a 69 anos**, `nps_score` varia de
**0 a 10** e as variáveis financeiras estão armazenadas como números decimais.

Nesta versão da base, os tipos carregados são coerentes com o significado das
variáveis. Por isso, não será feita conversão automática de tipos.

# BLOCO 02 - QUALIDADE DOS DADOS

## 1. Diagnóstico inicial de qualidade

O diagnóstico reúne três informações por coluna:

- Tipo de dado;
- Quantidade de valores ausentes;
- Quantidade de valores únicos.

Essa visão permite identificar rapidamente colunas que exigem investigação.

In [7]:
# Diagnóstico simples por coluna
diagnostico = pd.DataFrame({
    "tipos_dados": df.dtypes.astype(str),
    "quantidade_nulos": df.isnull().sum(),
    "valores_unicos": df.nunique(dropna=False)
})

diagnostico

,tipos_dados,quantidade_nulos,valores_unicos
customer_id,int64,0,2500
customer_age,int64,0,52
customer_region,object,0,5
customer_tenure_months,int64,0,119
order_id,int64,0,2500
order_value,float64,0,2457
items_quantity,int64,0,6
discount_value,float64,0,2050
payment_installments,int64,0,11
delivery_time_days,int64,0,13


### Resultado observado

O diagnóstico confirma que:

- Nenhuma coluna possui valores ausentes;
- `customer_id` e `order_id` possuem 2.500 valores únicos;
- `customer_region` possui 5 categorias;
- `repeat_purchase_30d` possui 2 valores possíveis;
- `nps_score` possui 101 valores diferentes, pois a base contém notas decimais.

Esses resultados serão detalhados nas próximas etapas.

## 2. Verificação de registros duplicados

São verificadas três situações:

1. Linhas completamente iguais;
2. `order_id` repetido;
3. `customer_id` repetido.

A verificação separada é importante porque identificadores diferentes representam
conceitos diferentes.

In [8]:
print("=" * 32)
print("Análise de registros duplicados")
print("=" * 32)


# Linhas completamente iguais
linhas_duplicadas = df.duplicated().sum()

# Identificadores de pedido repetidos
pedidos_duplicados = df["order_id"].duplicated().sum()

# Identificadores de cliente repetidos
clientes_duplicados = df["customer_id"].duplicated().sum()

print(f"Linhas completamente duplicadas: {linhas_duplicadas}")
print(f"order_id duplicados: {pedidos_duplicados}")
print(f"customer_id duplicados: {clientes_duplicados}")

Análise de registros duplicados
Linhas completamente duplicadas: 0
order_id duplicados: 0
customer_id duplicados: 0


## 3. Validação de regras de qualidade

Além dos tipos de dados, foram avaliadas regras de coerência relacionadas ao
significado de cada coluna.

As regras foram separadas em dois níveis:

- **Regra crítica:** indica uma relação incompatível com a definição das colunas.
  O registro é separado da base principal e mantido em um arquivo de auditoria;
- **Regra de atenção:** indica uma situação incomum, mas possível. O registro é
  mantido e recebe um indicador para investigação na EDA.

Essa abordagem evita corrigir valores sem evidência e mantém rastreabilidade sobre
as decisões adotadas.

In [9]:
print("=" * 25)
print("   Validação de regras")
print("=" * 25)

# Regras básicas de validação
regras_validacao = {
    "Idade menor que zero": df["customer_age"] < 0,
    "Idade maior que 120": df["customer_age"] > 120,
    "Tempo de relacionamento negativo": df["customer_tenure_months"] < 0,
    "Valor do pedido negativo": df["order_value"] < 0,
    "Quantidade de itens menor ou igual a zero": df["items_quantity"] <= 0,
    "Desconto negativo": df["discount_value"] < 0,
    "Tempo de entrega menor ou igual a zero": df["delivery_time_days"] <= 0,
    "Dias de atraso negativos": df["delivery_delay_days"] < 0,
    "Frete negativo": df["freight_value"] < 0,
    "Contatos com atendimento negativos": df["customer_service_contacts"] < 0,
    "Tempo de resolução negativo": df["resolution_time_days"] < 0,
    "NPS fora da escala de 0 a 10": ~df["nps_score"].between(0, 10),
    "Recompra fora das categorias 0 e 1": ~df["repeat_purchase_30d"].isin([0, 1]),
    "Quantidade de reclamações negativa": df["complaints_count"] < 0,
    "Frete menor que 0": df["freight_value"] <0
}

# Regras específicas relacionadas ao pedido e à entrega
regras_contexto = {
    "Atraso maior ou igual que o tempo total de entrega":
        df["delivery_delay_days"] >= df["delivery_time_days"],

    "Desconto maior que o valor do pedido":
        df["discount_value"] > df["order_value"],

    "Frete maior que o valor do pedido":
        df["freight_value"] > df["order_value"],
}

# Contagem de registros em cada regra
validacao_regras = pd.DataFrame({
    "quantidade": {
        **{nome: condicao.sum() for nome, condicao in regras_validacao.items()},
        **{nome: condicao.sum() for nome, condicao in regras_contexto.items()},
    }
})

validacao_regras["percentual"] = (
    validacao_regras["quantidade"] / len(df) * 100
).round(2)

validacao_regras

   Validação de regras


,quantidade,percentual
Idade menor que zero,0,0.00
Idade maior que 120,0,0.00
Tempo de relacionamento negativo,0,0.00
Valor do pedido negativo,0,0.00
Quantidade de itens menor ou igual a zero,0,0.00
Desconto negativo,0,0.00
Tempo de entrega menor ou igual a zero,0,0.00
Dias de atraso negativos,0,0.00
Frete negativo,0,0.00
Contatos com atendimento negativos,0,0.00


### Resultado observado

As regras básicas de domínio não identificaram valores negativos, idades fora do
intervalo adotado, NPS fora da escala ou categorias inválidas de recompra.

Entretanto, foram encontradas três situações que exigem avaliação específica:

- **267 registros** com atraso maior ou igual que o tempo total de entrega;
- **35 registros** com desconto maior que o valor do pedido;
- **18 registros** com frete maior que o valor do pedido.

## 4. Padronização dos nomes das colunas

Os nomes já estavam próximos de um padrão consistente. Mesmo assim, aplicamos
uma padronização simples para evitar espaços ou diferenças entre letras maiúsculas
e minúsculas.

O padrão adotado utiliza:

- Letras minúsculas;
- Remoção de espaços no início e no final;
- Sublinhado no lugar de espaços internos.

In [10]:
# Padronização dos nomes das colunas
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Conferência após a padronização
df.columns.tolist()

['customer_id',
 'customer_age',
 'customer_region',
 'customer_tenure_months',
 'order_id',
 'order_value',
 'items_quantity',
 'discount_value',
 'payment_installments',
 'delivery_time_days',
 'delivery_delay_days',
 'freight_value',
 'delivery_attempts',
 'customer_service_contacts',
 'resolution_time_days',
 'nps_score',
 'repeat_purchase_30d',
 'complaints_count',
 'csat_internal_score']

**4.1 Padronização de `customer_region`**

Categorias com diferenças de espaços ou grafia podem ser interpretadas como categorias distintas, por isso aplicamos uma padronização simples.

In [11]:
# Remove espaços, converte temporariamente para letras minúsculas e substitui pelas grafias definidas para o projeto.
df["customer_region"] = (
    df["customer_region"]
    .str.strip()
    .str.lower()
    .replace({
        "norte": "Norte",
        "nordeste": "Nordeste",
        "centro-oeste": "Centro-Oeste",
        "centro oeste": "Centro-Oeste",
        "sudeste": "Sudeste",
        "sul": "Sul"
    })
)

# Distribuição após a padronização
regioes_depois = df["customer_region"].value_counts(dropna=False)

regioes_depois

customer_region
Sul             521
Sudeste         520
Norte           506
Nordeste        485
Centro-Oeste    468
Name: count, dtype: int64

# BLOCO 03 - TRATAMENTO

Depois serão criadas as variáveis:

- `had_delivery_delay`
- `had_customer_service_contact `
- `had_complaint`
- `delivery_delay_ratio`
- `age_range`
- `customer_tenure_range`
- `order_value_range`
- `delivery_time_range`
- `complaints_range`
- `nps_class`
- `perc_discount_range`
- `delay_range`


## 1. Tratamento das regras de qualidade

Nesta etapa serão aplicadas as decisões definidas para:

- Atraso maior que o tempo total de entrega;
- Desconto maior que o valor do pedido;
- Frete maior que o valor do pedido.

A base original continuará preservada em `df_original`.

### 1.1 - Indicadores operacionais

**Indicador de atraso**

A variável `had_delivery_delay` foi criada para identificar se o pedido apresentou pelo
menos um dia de atraso na entrega.

A classificação utilizada foi:

- `0`: pedido sem atraso;
- `1`: pedido com atraso.

In [12]:
# Cria o indicador binário de atraso
df["had_delivery_delay"] = np.where(
    df["delivery_delay_days"] > 0,
    1,
    0
)

# Quantidade e percentual de pedidos em cada situação
resumo_atraso = pd.DataFrame({
    "quantidade": df["had_delivery_delay"].value_counts().sort_index(),
    "percentual": (
        df["had_delivery_delay"].value_counts(normalize=True).sort_index() * 100
    ).round(2)
})

resumo_atraso

,quantidade,percentual
had_delivery_delay,,
0,277,11.08
1,2223,88.92


**Indicador de contato com o atendimento**

A variável `had_customer_service_contact` foi criada para identificar se o cliente
precisou entrar em contato com a equipe de atendimento.

A classificação utilizada foi:

- `0`: nenhum contato registrado;
- `1`: pelo menos um contato registrado.

In [13]:
# Cria o indicador binário de atendimento
df["had_customer_service_contact"] = np.where(
    df["customer_service_contacts"] > 0,
    1,
    0
)

# Quantidade e percentual de pedidos em cada situação
resumo_atendimento = pd.DataFrame({
    "quantidade": df["had_customer_service_contact"].value_counts().sort_index(),
    "percentual": (
        df["had_customer_service_contact"].value_counts(normalize=True).sort_index() * 100
    ).round(2)
})

resumo_atendimento

,quantidade,percentual
had_customer_service_contact,,
0,554,22.16
1,1946,77.84


**Indicador de reclamação**

A variável `had_complaint` foi criada para identificar se existe pelo menos uma
reclamação registrada para o pedido.

A classificação utilizada foi:

- `0`: nenhuma reclamação registrada;
- `1`: uma ou mais reclamações registradas.

In [14]:
# Cria o indicador binário de reclamação
df["had_complaint"] = np.where(
    df["complaints_count"] > 0,
    1,
    0
)

# Quantidade e percentual de pedidos em cada situação
resumo_reclamacao = pd.DataFrame({
    "quantidade": df["had_complaint"].value_counts().sort_index(),
    "percentual": (
        df["had_complaint"].value_counts(normalize=True).sort_index() * 100
    ).round(2)
})

resumo_reclamacao

,quantidade,percentual
had_complaint,,
0,23,0.92
1,2477,99.08


### Síntese dos indicadores operacionais

Os três indicadores criados apresentam alta concentração na categoria `1`:

| Indicador | Quantidade com ocorrência | Percentual |
|---|---:|---:|
| Pedido com atraso | 2.223 | 88,92% |
| Contato com atendimento | 1.946 | 77,84% |
| Pedido com reclamação | 2.477 | 99,08% |

Os resultados mostram que atraso, atendimento e reclamação são eventos frequentes
na base analisada.

Entretanto, as frequências não demonstram que esses fatores causam redução no
NPS. Elas apenas orientam as hipóteses que deverão ser investigadas na EDA.

Também será necessário considerar que os grupos são desbalanceados, principalmente
no indicador de reclamação. Por isso, as variáveis numéricas originais devem ser
preservadas:

- `delivery_delay_days`;
- `customer_service_contacts`;
- `complaints_count`.

Essas colunas permitem análises mais detalhadas do que os indicadores binários.

### 1.2 - Regras críticas e registros para auditoria

Foram consideradas críticas as seguintes relações:

1. `delivery_delay_days >= delivery_time_days`

   O atraso não deve superar ou igualar o tempo **total** de entrega, considerando a definição apresentada no dicionário de dados.

2. `discount_value > order_value`

   O desconto não deve superar o valor total do pedido. Como a base não informa outro valor de referência para a compra, não existe informação suficiente para corrigir o registro com segurança.

A prática adotada será a **quarentena de dados**:

- Os registros não serão corrigidos por suposição;
- Eles serão retirados da base principal usada na EDA e na modelagem;
- Serão salvos separadamente com o motivo da inconsistência;
- A base original permanecerá preservada.

In [15]:
# Máscaras das regras críticas
regra_atraso = (
    df["delivery_delay_days"] >= df["delivery_time_days"]
)

regra_desconto = (
    df["discount_value"] > df["order_value"]
)

# Registro crítico = viola pelo menos uma das regras
mascara_registro_critico = regra_atraso | regra_desconto

# Registros que violam simultaneamente as duas regras
regra_ambas = regra_atraso & regra_desconto


# Resumo antes do tratamento
resumo_regras_criticas = pd.DataFrame({
    "regra": [
        "Atraso maior ou igual ao tempo total de entrega",
        "Desconto maior que o valor do pedido",
        "Violação simultânea das duas regras",
        "Pelo menos uma regra crítica"
    ],
    "quantidade": [
        regra_atraso.sum(),
        regra_desconto.sum(),
        regra_ambas.sum(),
        mascara_registro_critico.sum()
    ]
})

resumo_regras_criticas["percentual"] = (
    resumo_regras_criticas["quantidade"] / len(df) * 100
).round(1)

resumo_regras_criticas

,regra,quantidade,percentual
0,Atraso maior ou igual ao tempo total de entrega,267,10.7
1,Desconto maior que o valor do pedido,35,1.4
2,Violação simultânea das duas regras,4,0.2
3,Pelo menos uma regra crítica,298,11.9


**Identificação do motivo da inconsistência**

Os registros críticos serão mantidos em uma base separada.

A coluna `inconsistency_reason` informa qual regra foi violada. Quando o mesmo
pedido viola as duas regras, os dois motivos são registrados.

Esse arquivo permite revisar os casos posteriormente sem misturá-los à base
principal.

In [16]:
# Seleciona os registros que violam pelo menos uma regra crítica
df_registros_rejeitados = df.loc[mascara_registro_critico].copy()

# Registra o motivo de cada inconsistência
df_registros_rejeitados["inconsistency_reason"] = ""
# Cria coluna vazia inconsistency_reason

def registrar_inconsistencia(regra):
    motivos = []

    if regra_atraso[regra]:
        motivos.append("atraso_maior_que_tempo_entrega")

    if regra_desconto[regra]:
        motivos.append("desconto_maior_que_pedido")

    return "; ".join(motivos)

df_registros_rejeitados["inconsistency_reason"] = (
    df_registros_rejeitados.index.map(registrar_inconsistencia)
)

print(f"Registros separados para auditoria: {len(df_registros_rejeitados)}")

df_registros_rejeitados[
    [
        "order_id",
        "order_value",
        "discount_value",
        "delivery_time_days",
        "delivery_delay_days",
        "inconsistency_reason"
    ]
].head(10)

df_registros_rejeitados.to_csv("../data/processed/nps_registros_rejeitados.csv")

Registros separados para auditoria: 298


In [17]:
# Mantém na base principal somente os registros aprovados
df = df.loc[~mascara_registro_critico].copy()

print(f"Registros da base original: {len(df_original)}")
print(f"Registros separados para auditoria: {len(df_registros_rejeitados)}")
print(f"Registros mantidos na base tratada: {len(df)}")

print(
    "Conferência do total: "
    f"{len(df_registros_rejeitados) + len(df)}"
)



Registros da base original: 2500
Registros separados para auditoria: 298
Registros mantidos na base tratada: 2202
Conferência do total: 2500


### 1.3 - Frete maior que o valor do pedido

O frete maior que o valor do pedido é uma situação comercialmente incomum, mas não
é uma impossibilidade lógica.

Ela pode ocorrer em pedidos de baixo valor, rotas mais caras ou operações sem
subsídio de frete. Como a base não possui peso, distância, modalidade de entrega ou
regra de cobrança, não há evidência suficiente para classificar esses registros como
erro.

#### Decisão adotada

Os registros serão mantidos e receberão a variável
`freight_exceeds_order_value`:

- `0`: frete menor ou igual ao valor do pedido;
- `1`: frete maior que o valor do pedido.

A variável permitirá verificar na EDA se essa relação está associada a notas menores
de NPS.

In [18]:
# Cria um indicador para pedidos cujo frete supera o valor do pedido
df["freight_exceeds_order_value"] = np.where(
    df["freight_value"] > df["order_value"],
    1,
    0
)

resumo_frete_pedido = pd.DataFrame({
    "quantidade": df["freight_exceeds_order_value"].value_counts(),
    "percentual": (
        df["freight_exceeds_order_value"].value_counts(normalize=True) * 100
    ).round(2)
})

resumo_frete_pedido

,quantidade,percentual
freight_exceeds_order_value,,
0,2195,99.68
1,7,0.32


### 1.4 - Atraso percentual

Após a retirada dos registros que violavam a regra crítica de entrega, a variável
`delivery_delay_ratio` pode ser calculada de forma coerente:

`delivery_delay_days / delivery_time_days`

O resultado indica quanto do tempo total de entrega corresponde aos dias de atraso.

In [19]:
# Calcula o atraso proporcional ao tempo total de entrega
df["delivery_delay_ratio"] = (
    df["delivery_delay_days"] / df["delivery_time_days"]
)

# Confere se o resultado está entre 0 e 1
print(
    "Valores acima de 100%: ",
    (df["delivery_delay_ratio"] > 1).sum()
)

df["delivery_delay_ratio"].describe().round(2)

Valores acima de 100%:  0


count    2202.00
mean        0.27
std         0.20
min         0.00
25%         0.12
50%         0.23
75%         0.38
max         0.88
Name: delivery_delay_ratio, dtype: float64

#### Resultado do tratamento

A base principal passa a conter apenas registros que não ferem as duas regras críticas.

O tratamento não tentou adivinhar ou substituir valores. Os registros separados continuam disponíveis em um arquivo de auditoria.

O frete maior que o pedido foi mantido como uma característica observável. Essa decisão preserva uma experiência potencialmente importante para explicar a insatisfação, sem apresentar o caso como erro confirmado.

## 2 - Criação de variáveis - faixas

Faixas de valores para agrupar os registros em variaveis relevantes.

As novas colunas serão usadas para facilitar segmentações e comparações na EDA.

Os intervalos são uma decisão analítica deste projeto, após avaliação da base.

### 2.1 - Faixa etária

In [20]:
# Limites e rótulos das faixas etárias
limites_idade = [0, 24, 34, 44, 54, 64, np.inf]
#np.inf Serve para marcar o limite superior aberto da última faixa.

rotulos_idade = [
    "Até 24 anos",
    "25 a 34 anos",
    "35 a 44 anos",
    "45 a 54 anos",
    "55 a 64 anos",
    "65 anos ou mais"
]

# Criação da coluna faixa etária
df["age_range"] = pd.cut(
    df["customer_age"],
    bins=limites_idade,
    labels=rotulos_idade,
    include_lowest=True
)

# Distribuição das faixas
contagem_faixas = df["age_range"].value_counts(sort=False, dropna=False).to_frame("quantidade")
#calcular percentual
contagem_faixas["percentual"] = (contagem_faixas["quantidade"]/len(df)*100).round(1)

contagem_faixas.reset_index()


,age_range,quantidade,percentual
0,Até 24 anos,307,13.9
1,25 a 34 anos,406,18.4
2,35 a 44 anos,441,20.0
3,45 a 54 anos,427,19.4
4,55 a 64 anos,416,18.9
5,65 anos ou mais,205,9.3


#### Resultado observado

Todos os registros receberam uma faixa etária. A maior faixa é **35 a 44 anos**, com 509 registros (20%). A menor é **65 anos ou mais**, com 238 registros (9,5%). As demais faixas também possuem quantidade suficiente para comparações exploratórias.

### 2.2 - Faixa de tempo de relacionamento

In [21]:
# Limites e rótulos das faixas de relacionamento
limites_tenure = [0, 12, 24, 48, 72, np.inf]

rotulos_tenure = [
    "Até 12 meses",
    "13 a 24 meses",
    "25 a 48 meses",
    "49 a 72 meses",
    "Mais de 72 meses"
]

# Criação da faixa de relacionamento
df["customer_tenure_range"] = pd.cut(
    df["customer_tenure_months"],
    bins=limites_tenure,
    labels=rotulos_tenure,
    include_lowest=True
)

# Distribuição das faixas
tempo_relacionamento = df["customer_tenure_range"].value_counts(sort=False, dropna=False).to_frame("quantidade")
tempo_relacionamento["percentual"] = (tempo_relacionamento["quantidade"]/len(df)*100).round(1)

tempo_relacionamento.reset_index()


,customer_tenure_range,quantidade,percentual
0,Até 12 meses,213,9.7
1,13 a 24 meses,224,10.2
2,25 a 48 meses,413,18.8
3,49 a 72 meses,435,19.8
4,Mais de 72 meses,917,41.6


#### Resultado observado

Todos os registros receberam uma faixa de relacionamento. O maior grupo é **Mais de 72 meses**, com 1.041 registros, que representa 42% da basse. Isso mostra que uma parte relevante da base possui relacionamento mais longo com a empresa.

### 2.3 - Faixa de valor do pedido

In [22]:
limites_order_value = [0, 150, 300, 500, 800, np.inf]

rotulos_order_value = [
    "Até R$150",
    "R$150 a R$300",
    "R$300 a R$500",
    "R$500 a R$800",
    "Acima de R$800"
]

# Criação da faixa de preço
df["order_value_range"] = pd.cut(
    df["order_value"],
    bins=limites_order_value,
    labels=rotulos_order_value,
    include_lowest=True
)

# Distribuição das faixas
valor_pedido = df["order_value_range"].value_counts(sort=False, dropna=False).to_frame("quantidade")
valor_pedido["percentual"] = (valor_pedido["quantidade"]/len(df)*100).round(1)

valor_pedido.reset_index()


,order_value_range,quantidade,percentual
0,Até R$150,276,12.5
1,R$150 a R$300,558,25.3
2,R$300 a R$500,644,29.2
3,R$500 a R$800,484,22.0
4,Acima de R$800,240,10.9


### 2.4 - Faixa de tempo de entrega
Temos que o minimo é 2 e o maximo é 14 dias.

In [23]:
limites_delivery_time = [0, 5, 8, 11, np.inf]

rotulos_delivery_time = [
    "Entrega de 0 a 5 dias",
    "Entrega de 5 a 8 dias",
    "Entrega de 8 a 11 dias",
    "Entrega 11+ dias"
]

# Criação da faixa de tempo de entrega
df["delivery_time_range"] = pd.cut(
    df["delivery_time_days"],
    bins=limites_delivery_time,
    labels=rotulos_delivery_time,
    include_lowest=True
)

# Distribuição das faixas
tempo_entrega = df["delivery_time_range"].value_counts(sort=False, dropna=False).to_frame("quantidade")
tempo_entrega["percentual"] = (tempo_entrega["quantidade"]/len(df)*100).round(1)

tempo_entrega.reset_index()

,delivery_time_range,quantidade,percentual
0,Entrega de 0 a 5 dias,500,22.7
1,Entrega de 5 a 8 dias,556,25.2
2,Entrega de 8 a 11 dias,549,24.9
3,Entrega 11+ dias,597,27.1


### 2.5 - Faixa de numero de reclamações

In [24]:
limites_complaints = [0, 2, 4, 6, 12]

rotulos_complaints = [
    "De 0 a 2 Reclamações",
    "De 2 a 4 Reclamações",
    "De 4 a 6 Reclamações",
    "6 ou mais Reclamações"
]

# Criação da faixa de reclamações
df["complaints_range"] = pd.cut(
    df["complaints_count"],
    bins=limites_complaints,
    labels=rotulos_complaints,
    include_lowest=True
)

# Distribuição das faixas
reclamacoes = df["complaints_range"].value_counts(sort=False, dropna=False).to_frame("quantidade")
reclamacoes["percentual"] = (reclamacoes["quantidade"]/len(df)*100).round(1)

reclamacoes.reset_index()

,complaints_range,quantidade,percentual
0,De 0 a 2 Reclamações,390,17.7
1,De 2 a 4 Reclamações,967,43.9
2,De 4 a 6 Reclamações,617,28.0
3,6 ou mais Reclamações,228,10.4


### 2.6 - Classificação do NPS (variável alvo)

A base possui notas decimais entre 0 e 10. Para que todos os valores recebam uma
classe, adotamos intervalos contínuos:

- **Detrator:** nota menor ou igual a 6;
- **Neutro:** nota maior que 6 e menor ou igual a 8;
- **Promotor:** nota maior que 8 e menor ou igual a 10.

Essa é uma adaptação operacional da regra proposta no escopo. A nota original será
preservada.

In [25]:
# Conferência dos limites da variável antes da classificação
print(f"Menor nota de NPS: {df['nps_score'].min()}")
print(f"Maior nota de NPS: {df['nps_score'].max()}")

# Função de classificação do NPS
def classificar_nps(nota):
    if pd.isna(nota):
        return np.nan
    elif nota < 7:
        return "Detrator"
    elif nota < 9:
        return "Neutro"
    else:
        return "Promotor"

# Criação da classe
df["nps_class"] = df["nps_score"].apply(classificar_nps)

# Quantidade e percentual por classe
resumo_classes_nps = pd.DataFrame({
    "quantidade": df["nps_class"].value_counts(),
    "percentual": (
        df["nps_class"].value_counts(normalize=True) * 100
    ).round(1)
})

resumo_classes_nps.reset_index()

Menor nota de NPS: 0.0
Maior nota de NPS: 10.0


,nps_class,quantidade,percentual
0,Detrator,1829,83.1
1,Neutro,267,12.1
2,Promotor,106,4.8


### Resultado observado

A classificação gerou:

- 1.964 Detrator (**83,7**);
- 274 Neutro (**11,7%**);
- 108 Promotor (**4,6%**).

A base apresenta forte concentração na classe Detrator. Esse desbalanceamento
deverá ser considerado na etapa de modelagem, mas o tratamento específico pertence
ao notebook do item 4.

Para as próximas etapas:

- `nps_score` poderá ser utilizado como alvo de regressão;
- `nps_category` poderá ser utilizada como alvo de classificação.

### 2.7 - Variável de % de desconto

In [26]:
df["discount_percentage"] = (df["discount_value"] / df["order_value"] * 100).round(1)

limites_discount_pct = [0, 5, 10, 15, 25, np.inf]

rotulos_discount_pct = [
    "Sem/Min Desconto (0-5%)",
    "Desconto Baixo (5-10%)",
    "Desconto Moderado (10-15%)",
    "Desconto Alto (15-25%)",
    "Desconto Excessivo (25%+)"
]

# Criação da faixa de reclamações
df["perc_discount_range"] = pd.cut(
    df["discount_percentage"],
    bins=limites_discount_pct,
    labels=rotulos_discount_pct,
    include_lowest=True
)

# Distribuição das faixas
df_discount_range = df["perc_discount_range"].value_counts(sort=False, dropna=False).to_frame("quantidade")
df_discount_range["percentual"] = (df_discount_range["quantidade"]/len(df)*100).round(1)
df_discount_range.reset_index()


,perc_discount_range,quantidade,percentual
0,Sem/Min Desconto (0-5%),1019,46.3
1,Desconto Baixo (5-10%),495,22.5
2,Desconto Moderado (10-15%),241,10.9
3,Desconto Alto (15-25%),225,10.2
4,Desconto Excessivo (25%+),222,10.1


### 2.8 - Variável de % de tempo de atraso

In [27]:
df["delay_percentage"] = (df["delivery_delay_days"] / df["delivery_time_days"] * 100).round(2)
# quantos % de atraso teve na entrega em relação ao total de dias de entrega\

limites_delay_dias = [0, 1, 2, 3, 6, np.inf]

rotulos_delay_dias = [
    "No Prazo (0 dias)",
    "Atraso Leve (1 dia)",
    "Atraso Moderado (2 dias)",
    "Atraso Significativo (3 dias)",
    "Atraso Severo (6+ dias)"
]
# Criação da faixa de reclamações
df["delay_range"] = pd.cut(
    df["delivery_delay_days"],
    bins=limites_delay_dias,
    labels=rotulos_delay_dias,
    include_lowest=True
)

# Distribuição das faixas
df_delay_range = df["delay_range"].value_counts(sort=False, dropna=False).to_frame("quantidade")
df_delay_range["percentual"] = (df_delay_range["quantidade"]/len(df)*100).round(1)
df_delay_range.reset_index()


,delay_range,quantidade,percentual
0,No Prazo (0 dias),879,39.9
1,Atraso Leve (1 dia),572,26.0
2,Atraso Moderado (2 dias),439,19.9
3,Atraso Significativo (3 dias),303,13.8
4,Atraso Severo (6+ dias),9,0.4


## 3 - Exportação da base tratada

A base tratada será salva na pasta `data/processed` para ser utilizada na etapa de EDA.


In [28]:
df.to_excel("../data/processed/nps_tratado.xlsx")